# 01 — Data Quality, Validation & Exploratory Analysis

## Objective
This notebook performs the complete **data-quality and exploratory-data-analysis stage** for the Hillstrom email experiment.

It covers:
- dataset discovery and loading
- schema and datatype inspection
- missing values and duplicates
- invalid-value checks
- treatment-arm validation
- descriptive statistics
- treatment-group EDA
- outcome distributions
- initial business observations

No treatment/outcome model is trained here.

In [ ]:
from pathlib import Path
import sys, json, warnings
warnings.filterwarnings("ignore")

# Find the project root so the notebook works whether launched from the project root
# or from inside the notebooks directory.
_here = Path.cwd().resolve()
_candidates = [_here, *_here.parents]
ROOT = next((p for p in _candidates if (p / "src" / "config.py").exists()), _here)
if not (ROOT / "src" / "config.py").exists():
    raise FileNotFoundError("Could not locate project root containing src/config.py")

sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)

In [ ]:
from src.data_utils import load_data, validate_data, prepare, save_validation
from src.config import DATA_PATH, TREATMENTS

print("Dataset path:", DATA_PATH)
df_raw = load_data(DATA_PATH)
print("Raw shape:", df_raw.shape)
display(df_raw.head())

In [ ]:
report = validate_data(df_raw)
print(json.dumps(report, indent=2, default=str))

save_validation(report)
print("\nValidation report saved to:", ROOT / "artifacts" / "metadata" / "data_quality.json")

## 1. Schema and data types

In [ ]:
display(df_raw.dtypes.to_frame("dtype"))
display(df_raw.describe(include="all").T)

## 2. Missing values and duplicates

In [ ]:
missing = df_raw.isna().sum().sort_values(ascending=False)
display(missing[missing > 0].to_frame("missing_count"))
print("Duplicate rows:", int(df_raw.duplicated().sum()))

## 3. Treatment assignment

In [ ]:
treatment_counts = df_raw["segment"].value_counts(dropna=False)
display(treatment_counts.to_frame("customers"))
display((treatment_counts / len(df_raw) * 100).round(2).to_frame("percent"))
assert set(TREATMENTS).issubset(set(df_raw["segment"].unique()))

## 4. Binary outcome/value checks

In [ ]:
for col in ["mens", "womens", "newbie", "visit", "conversion"]:
    vals = sorted(df_raw[col].dropna().unique().tolist())
    print(f"{col}: unique values = {vals}")
    assert set(vals).issubset({0, 1}), f"Unexpected values in {col}: {vals}"

## 5. Prepared analysis data

In [ ]:
df = prepare(df_raw)
print("Prepared shape:", df.shape)
display(df.head())

## 6. Experiment-group descriptive statistics

In [ ]:
group_summary = (
    df.groupby("segment")
      .agg(
          customers=("segment", "size"),
          visit_rate=("visit", "mean"),
          conversion_rate=("conversion", "mean"),
          avg_spend=("spend", "mean"),
          median_spend=("spend", "median"),
          avg_recency=("recency", "mean"),
          avg_history=("history", "mean"),
      )
      .reset_index()
)
display(group_summary.style.format({
    "visit_rate":"{:.2%}", "conversion_rate":"{:.2%}",
    "avg_spend":"${:.2f}", "median_spend":"${:.2f}",
    "avg_recency":"{:.2f}", "avg_history":"${:.2f}"
}))
group_summary.to_csv(ROOT / "artifacts" / "metrics" / "notebook_01_group_summary.csv", index=False)

## 7. EDA visualizations

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

fig_dir = ROOT / "artifacts" / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)

fig, ax = plt.subplots(figsize=(8, 5))
df.groupby("segment")["conversion"].mean().plot(kind="bar", ax=ax)
ax.set_title("Conversion Rate by Experiment Arm")
ax.set_ylabel("Conversion rate")
ax.set_xlabel("Treatment")
ax.tick_params(axis="x", rotation=0)
fig.tight_layout()
fig.savefig(fig_dir / "notebook_01_conversion_by_arm.png", dpi=160)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
df.groupby("segment")["visit"].mean().plot(kind="bar", ax=ax)
ax.set_title("Visit Rate by Experiment Arm")
ax.set_ylabel("Visit rate")
ax.set_xlabel("Treatment")
ax.tick_params(axis="x", rotation=0)
fig.tight_layout()
fig.savefig(fig_dir / "notebook_01_visit_by_arm.png", dpi=160)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for arm, part in df.groupby("segment"):
    ax.hist(part["spend"], bins=40, alpha=0.45, label=arm)
ax.set_title("Spend Distribution by Experiment Arm")
ax.set_xlabel("Spend")
ax.set_ylabel("Customers")
ax.legend()
fig.tight_layout()
fig.savefig(fig_dir / "notebook_01_spend_distribution.png", dpi=160)
plt.show()

## Conclusion
This stage establishes whether the uploaded file is structurally suitable for the causal experiment analysis.

The next notebook uses the randomized treatment assignment to estimate **average treatment effects (ATE)** for visits, conversions, and spend.